In [1]:
import os
print(os.getpid())

221403


In [2]:
import liana as li
print(li.__version__)

1.7.1


In [3]:
import inspect
print(inspect.signature(li.method.cellphonedb.by_sample))

(adata: 'an.AnnData | MuData', sample_key: 'str', key_added: 'str' = 'liana_res', inplace: 'bool' = True, verbose: 'bool' = False, **kwargs)


In [4]:
import inspect
print(inspect.signature(li.method.cellphonedb))
# we can see n_jobs:1 so by default it accept cpus with different numbers

(adata: 'an.AnnData | MuData', groupby: 'str', resource_name: 'str' = 'consensus', expr_prop: 'float' = 0.1, min_cells: 'int' = 5, groupby_pairs: 'DataFrame | None' = None, base: 'float' = np.float64(2.718281828459045), supp_columns: 'list | None' = None, return_all_lrs: 'bool' = False, key_added: 'str' = 'liana_res', use_raw: 'bool | None' = True, layer: 'str | None' = None, de_method: 'str' = 't-test', n_perms: 'int' = 1000, seed: 'int' = 1337, n_jobs: 'int' = 1, resource: 'DataFrame | None' = None, interactions: 'list | None' = None, spatial_key: 'str' = 'spatial', spatial_kwargs: 'dict | None' = None, mdata_kwargs: 'dict | None' = None, inplace: 'bool' = True, verbose: 'bool | None' = False)


In [5]:
import os
print(os.cpu_count())

72


Jupyter kernel sees 72 CPUs. That doesn't necessarily mean you are allocated all 72, but it's enough to test parallelization.

In [6]:
# actual memory availability
import psutil

mem = psutil.virtual_memory()

print(f"Total RAM: {mem.total / 1024**3:.1f} GB")
print(f"Available RAM: {mem.available / 1024**3:.1f} GB")
print(f"Used RAM: {mem.used / 1024**3:.1f} GB")

Total RAM: 1510.3 GB
Available RAM: 1248.9 GB
Used RAM: 259.1 GB


In [7]:
# check whether your current CellPhoneDB process is actually using CPU

import os
import psutil

p = psutil.Process(os.getpid())

print("PID:", p.pid)
print("CPU %:", p.cpu_percent(interval=3))
print("Memory GB:", p.memory_info().rss / 1024**3)

PID: 221403
CPU %: 0.0
Memory GB: 0.29473876953125


current Jupyter kernel is using essentially no CPU, but that doesn't necessarily tell us whether the CellPhoneDB calculation is running in another process. More importantly, the calculation may have already stopped/hung.

In [9]:
# check the Jupyter kernel's process children.

import psutil

p = psutil.Process()

children = p.children(recursive=True)

[(c.pid, c.status(), c.cpu_percent(interval=0.5), 
  c.memory_info().rss / 1024**3) for c in children]

[]

[] means your Jupyter Python process currently has no child processes.

So LIANA is not currently running as a separate parallel worker process. We need to determine whether the calculation is actually still running or has stalled.

In [10]:
p = psutil.Process()

print("Status:", p.status())
print("CPU:", p.cpu_percent(interval=10))

Status: running
CPU: 0.0


In [11]:
# check cpu usage again

print(psutil.Process().cpu_percent(interval=5))

0.0
